Inports

In [1]:
# pip install pyodbc

In [2]:
# pip install -U imbalanced-learn

In [3]:
# pip install -U scikit-learn

In [4]:
#pip install mlxtend  

In [1]:
import pandas as pd
#Ig
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest
# Chi square
from sklearn.feature_selection import chi2
import numpy as np
# SU
import math
from scipy.stats import entropy
from sklearn import metrics

In [2]:
class ModelResult:
    def __init__(self, level_measure,project,b,rank,model_selection,intance_reduction,model,precision,recall,f,accuracy,count_metric,count_row):
        self.level_measure = level_measure
        self.project = project
        self.b = b
        self.rank = rank
        self.model_selection = model_selection
        self.intance_reduction = intance_reduction
        self.model = model
        self.precision = precision
        self.recall = recall
        self.f = f
        self.accuracy = accuracy
        self.count_metric = count_metric
        self.count_row = count_row
class SelectMeasure:
    def __init__(self,level_measure,project,b,rank,cs_su,ig_su,rf_su,cs_cos,ig_cos,rf_cos):
        self.level_measure = level_measure
        self.project = project
        self.b = b
        self.rank = rank
        self.cs_su = cs_su
        self.ig_su = ig_su
        self.rf_su = rf_su
        self.cs_cos = cs_cos
        self.ig_cos = ig_cos
        self.rf_cos = rf_cos

In [3]:
def GetMetrics(ytest, ypred,name):
    return pd.DataFrame([
        round(metrics.precision_score(ytest, ypred, average='weighted'), 2),
        round(metrics.recall_score(ytest, ypred, average='weighted'), 2),
        round(metrics.f1_score(ytest, ypred, average='weighted'), 2),
        round(metrics.accuracy_score(ytest, ypred), 2)
    ],
                    index=["pr","re","f","Ac"],
                    columns=[name])

In [4]:
_scoring = {'acc': 'accuracy',
           'prec_macro': 'precision_macro',
           'rec_micro': 'recall_macro',
           'f':'f1_macro'
           }
def GetMetricsCross(results,name):
    return pd.DataFrame([
        round(results['test_prec_macro'].mean(), 2),
        round(results['test_rec_micro'].mean(), 2),
        round(results['test_f'].mean(), 2),
        round(results['test_acc'].mean(), 2)
    ],
                    index=["pr","re","f","Ac"],
                    columns=[name])

In [5]:
import os
def FileList(directory,list=[]):
  for filename in os.listdir(directory):
      f = os.path.join(directory, filename)
      if os.path.isdir(f):
        FileList(f,list)
      # checking if it is a file
      if os.path.isfile(f) and os.path.splitext(f)[-1].lower() == '.csv':
        list.append(f)
  return np.array(list)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix

def NaiveBayes(X,y,name):
  model = MultinomialNB()
  return GetMetricsCross(cross_validate(model, X, y, scoring=_scoring,
                         cv=5, return_train_score=True),name)

In [7]:
from sklearn.neighbors import KNeighborsClassifier
def KNN(X,y,name):
  knn = KNeighborsClassifier(n_neighbors=5)
  return GetMetricsCross(cross_validate(knn, X, y, scoring=_scoring,
                         cv=5, return_train_score=True),name)

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
def RandomForest(X,y,name):
  clf = RandomForestClassifier(max_depth=2, random_state=0)
  return GetMetricsCross(cross_validate(clf, X, y, scoring=_scoring,
                         cv=5, return_train_score=True),name)

In [9]:
from sklearn.linear_model import LogisticRegression
def LogisticRegr(X,y,name):
  logreg = LogisticRegression(random_state=16)
  return GetMetricsCross(cross_validate(logreg, X, y, scoring=_scoring,
                         cv=5, return_train_score=True),name)

In [10]:
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
def SGD(X,y,name):
  clf = make_pipeline(StandardScaler(), SGDClassifier())
  return GetMetricsCross(cross_validate(clf, X, y, scoring=_scoring,
                         cv=5, return_train_score=True),name)

In [11]:
from sklearn.tree import DecisionTreeClassifier # Import Decision Tree Classifier
def DecisionTree(X,y,name):
  clf = DecisionTreeClassifier()
  return GetMetricsCross(cross_validate(clf, X, y, scoring=_scoring,
                         cv=5, return_train_score=True),name)

In [12]:
from sklearn import svm
from sklearn.model_selection import cross_validate
def SVG(X,y,name):
    clf = svm.SVC(kernel='linear', C=1, random_state=42)
    return GetMetricsCross(cross_validate(clf, X, y, scoring=_scoring,
                         cv=5, return_train_score=True),name)

In [13]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

def ExecuteModel(model,IntanceReduction,X,y,column):
  try:
    if IntanceReduction == "RUS":
       rus = RandomUnderSampler(random_state=0)
       X, y = rus.fit_resample(X, y)
    elif IntanceReduction == "ROS":
       ros = RandomOverSampler(random_state=0)
       X, y = ros.fit_resample(X, y)

    # Algoritmo a ejecutar
    if model == "NB":
      return NaiveBayes(X,y,column)
    elif model == "KNN":
      return KNN(X,y,column)
    elif model == "DT":
      return DecisionTree(X,y,column)
    elif model == "SGD":
      return SGD(X,y,column)
    elif model == "LR":
      return LogisticRegr(X,y,column)
    elif model == "RF":
      return RandomForest(X,y,column)
    elif model == "SVG":
      return SVG(X,y,column)
  except Exception as e:
     print(str(e))
     return pd.DataFrame([-1,-1,-1,-1],
                    index=["pr","re","f","Ac"],
                    columns=[column])


In [14]:
import pyodbc 
def insertDb(data):
    conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=JORDANYS;'
                      'Database=magister;'
                      'Trusted_Connection=yes;')
    cursor = conn.cursor()
  
    dataInsert = "'{level_measure}','{project}','{b}','{rank}','{model_selection}','{intance_reduction}','{model}','{precision}','{recall}','{f}','{accuracy}','{count_metric}','{count_row}'".format(
        level_measure = data.level_measure ,
        project = data.project,
        b = data.b,
        rank = data.rank,
        model_selection = data.model_selection,
        intance_reduction = data.intance_reduction,
        model = data.model,
        precision = data.precision,
        recall = data.recall,
        f = data.f,
        accuracy = data.accuracy,
        count_metric = data.count_metric,
        count_row = data.count_row
    )
 
    cursor.execute('''
                INSERT INTO model_results_cross ([level_measure],[project],[b],[rank],[model_selection],[intance_reduction],[model],[precision],[recall],[f],[accuracy],[count_metric],[count_row])
                VALUES
                ('''+ dataInsert+''')

                ''')

    conn.commit()

In [15]:
def SelectMetricsDb(data):
    conn = pyodbc.connect('Driver={SQL Server};'
                      'Server=JORDANYS;'
                      'Database=magister;'
                      'Trusted_Connection=yes;')
    cursor = conn.cursor()
   
    query = "SELECT DISTINCT metric FROM [magister].[dbo].[select_measure] where b = '{b}' and rank = '{rank}' and project = '{project}' and level_measure = '{level_measure}' and cs_su like '%{cs_su}%' and ig_su like '%{ig_su}%' and rf_su like '%{rf_su}%' and cs_cos like '%{cs_cos}%' and ig_cos like '%{ig_cos}%' and rf_cos like '%{rf_cos}%'".format(
        level_measure = data.level_measure,
        project = data.project,
        b = data.b,
        rank = data.rank,
        cs_su = data.cs_su,
        ig_su = data.ig_su,
        rf_su = data.rf_su,
        cs_cos = data.cs_cos,
        ig_cos = data.ig_cos,
        rf_cos = data.rf_cos
    )
    cursor.execute(query)
    results = cursor.fetchall()
    return [i[0] for i in results] 

In [16]:
directory = 'C:/Users/danyr/Downloads/Python/full/'
listFile = FileList(directory)
projects =np.unique([os.path.split(os.path.dirname(i))[1] for i in listFile])
print(projects)

['Android-Universal-Image-Loader' 'BroadleafCommerce' 'MapDB' 'all'
 'antlr4' 'ceylon-ide-eclipse' 'elasticsearch' 'hazelcast' 'junit' 'mcMMO'
 'mct' 'neo4j' 'netty' 'orientdb' 'oryx' 'titan']


In [17]:
import csv
def insertFile(path,mode,row):
    with open(path, mode, newline='', encoding='UTF8') as f:
        # create the csv writer
        writer = csv.writer(f)
        # write a row to the csv file
        writer.writerow(row)
        

In [18]:
def insertDbCSV(data,file):
        insertFile(file,'a+',[
            data.level_measure
            ,data.project
            ,data.b
            ,data.rank
            ,data.model_selection
            ,data.intance_reduction
            ,data.model
            ,data.precision
            ,data.recall
            ,data.f
            ,data.accuracy
            ,data.count_metric
            ,data.count_row
        ]) 

In [19]:
def ModelsByCondition(project):
    file = 'D:/cross/'+project + '.csv'
    insertFile(file,'w', ['level_measure','project','b','rank','model_selection','intance_reduction','model','precision','recall','f','accuracy','count_metric','count_row'])  
    ranks = [0.5,0.7,0.9]
    bs = [0.5,0.7,0.9]
    models = ['SGD','LR','RF','NB','KNN','DT','SVG']
    modelsSelections = ['ALL','CS_SU','IG_SU','RF_SU','CS_COS','IG_COS','RF_COS']
    IntanceReductions = ['RUS','ROS','NONE']
    for rank in ranks:
          for b in bs:
                paths = [i for i in listFile if os.path.split(os.path.dirname(i))[1] == project] 
                for path in paths:
                    data=pd.read_csv(path)
                    try:
                      data=data.drop(columns=['Project','Hash', 'LongName','Parent'],axis=1, errors='ignore')
                      X =data.drop(labels=['Number of Bugs'], axis=1).astype(float).abs()
                      y =  [1 if i > 0 else 0 for i in data['Number of Bugs']]
                    except Exception as e:
                      print("Error in " + file_location + str(e))
                      continue
                    for model in models:
                        for modelsSelection in modelsSelections:
                            for reduction in IntanceReductions:
#                               data = pd.DataFrame([])
                              level = os.path.split(os.path.basename(path))[1].split('.')[0] 
                              measure = SelectMeasure(level,project,b,rank,'','','','','','')
                              if modelsSelection == 'CS_SU':
                                    measure.cs_su = 1
                              elif modelsSelection == 'IG_SU':
                                    measure.ig_su = 1
                              elif modelsSelection == 'RF_SU':
                                    measure.rf_su = 1
                              elif modelsSelection == 'CS_COS':
                                    measure.cs_cos = 1
                              elif modelsSelection == 'IG_COS':
                                    measure.ig_cos = 1
                              elif modelsSelection == 'RF_COS':
                                    measure.rf_cos = 1

                              metricsList = SelectMetricsDb(measure) 
                             
                              result = ExecuteModel(model,reduction,X[metricsList],y,modelsSelection)
                              insertDbCSV(ModelResult(level,project,b,rank,modelsSelection,reduction,model,result[modelsSelection]['pr'],result[modelsSelection]['re'],result[modelsSelection]['f'],result[modelsSelection]['Ac'],len(metricsList),X.shape[0]),file)


In [ ]:
# for project in projects:
#     ModelsByCondition(project)
from joblib import Parallel, delayed
Parallel(n_jobs=10)(delayed(ModelsByCondition)(project) for project in projects)

In [ ]:
# from sklearn.datasets import load_iris
# iris = load_iris()
# X = pd.DataFrame(iris['data'], columns=iris['feature_names'])
# y = pd.DataFrame(iris['target'], columns=['target'])
# print(SVG(X,y,'sdsd'))
# print(SGD(X,y,'ss'))
# print(LogisticRegr(X,y,'ss'))
# print(DecisionTree(X,y,'ss'))
# print(RandomForest(X,y,'ss'))
# print(KNN(X,y,'ss'))
# print(NaiveBayes(X,y,'ss'))